# [STARTER] Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [28]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [29]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set"
assert os.getenv("OPENAI_BASE_URL"), "OPENAI_BASE_URL is not set"

In [30]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

In [31]:
# TODO: Create a .env file with the following variables
# OPENAI_API_KEY="YOUR_KEY"
# CHROMA_OPENAI_API_KEY="YOUR_KEY"
# TAVILY_API_KEY="YOUR_KEY"

In [32]:
# TODO: Load environment variables
load_dotenv()

True

### VectorDB Instance

In [33]:
# TODO: Instantiate your ChromaDB Client
# Choose any path you want
chroma_client = chromadb.PersistentClient(path="chromadb")
#print(chroma_client.heartbeat())

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


### Collection

In [34]:
# TODO: Pick one embedding function
# If picking something different than openai, 
# make sure you use the same when loading it
embedding_fn = embedding_functions.OpenAIEmbeddingFunction()

In [35]:
# TODO: Create a collection
# Choose any name you want
collection = chroma_client.get_or_create_collection(
    name="udaplay",
    embedding_function=embedding_fn
)

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


### Add documents

In [36]:
# Make sure you have a directory "project/starter/games"
data_dir = "games"

for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue

    file_path = os.path.join(data_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        game = json.load(f)

    # You can change what text you want to index
    content = f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - {game['Description']}"

    # Use file name (like 001) as ID
    doc_id = os.path.splitext(file_name)[0]

    collection.add(
        ids=[doc_id],
        documents=[content],
        metadatas=[game]
    )

Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


### Semantic Search

In [39]:
test_queries = [
    "When was Pokemon Gold and Silver released?",
    "What are some good racing games on PlayStation?",
    "First 3D platformer Mario game?"
]

for query in test_queries:
    print(f"Query: {query}")
    print("-" * 50)

    results = collection.query(
        query_texts=[query],
        n_results=3
    )

    for i, metadata in enumerate(results["metadatas"][0]):
        print(f"Result {i+1}:")
        print(f" Name: {metadata.get('Name')}")
        print(f" Platform: {metadata.get('Platform')}")
        print(f" Year: {metadata.get('YearOfRelease')}")
        print(f" Description: {metadata.get('Description', '')[:100]}...")

    print()

Query: When was Pokemon Gold and Silver released?
--------------------------------------------------
Result 1:
 Name: Pokémon Gold and Silver
 Platform: Game Boy Color
 Year: 1999
 Description: Second-generation Pokémon games introducing new regions, Pokémon, and gameplay mechanics....
Result 2:
 Name: Pokémon Ruby and Sapphire
 Platform: Game Boy Advance
 Year: 2002
 Description: Third-generation Pokémon games set in the Hoenn region, featuring new Pokémon and double battles....
Result 3:
 Name: Super Mario 64
 Platform: Nintendo 64
 Year: 1996
 Description: A groundbreaking 3D platformer that set new standards for the genre, featuring Mario's quest to resc...

Query: What are some good racing games on PlayStation?
--------------------------------------------------
Result 1:
 Name: Gran Turismo 5
 Platform: PlayStation 3
 Year: 2010
 Description: A comprehensive racing simulator featuring a vast selection of vehicles and tracks, with realistic d...
Result 2:
 Name: Gran Turismo
 Platf